# DNR GT + Meta-Learning Training Notebook
## Structural Graph Transformer Training and Pandapower Comparison

Run `01_DNR_Data_Generation.ipynb` first so `dnr_diagnostic.h5` exists.
This notebook focuses on model work:

1. Load the generated diagnostic HDF5 dataset
2. Build structural graph tensors for the IEEE 33-bus feeder
3. Train the GT with fast supervised + Reptile-style updates
4. Compare guarded GT switching against BFS and pandapower


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# All dependencies for IEEE 33-bus DNR diagnostic pipeline
# ═══════════════════════════════════════════════════════════════════
import os, time, warnings, copy
from collections import Counter
from itertools import combinations
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import numba as nb
from scipy.stats import qmc
from scipy.special import erfinv
from scipy.stats import norm as sp_norm
from scipy.linalg import cholesky

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm

# ── Optional torch import (needed for loss scale check only) ──
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_OK = True
    print(f'PyTorch      : {torch.__version__}')
except ImportError:
    TORCH_OK = False
    print('PyTorch not found — loss scale check will be skipped')

print(f'NumPy        : {np.__version__}')
print(f'Pandas       : {pd.__version__}')
print(f'Numba        : {nb.__version__}')
print('All imports OK.')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — CONFIGURATION
# All hyperparameters in one place.
# Change EXCEL_FILE to match your local filename.
# ═══════════════════════════════════════════════════════════════════

# ── File paths ────────────────────────────────────────────────────
EXCEL_FILE  = 'Data_33bus.xlsx'       # IEEE 33-bus input data
DIAG_FILE   = 'dnr_diagnostic.h5'    # stores diagnostic results
DIAG_N      = 2000                   # scenarios for diagnostic run

# ── Network parameters (IEEE 33-bus standard) ─────────────────────
VN_KV    = 12.66    # nominal voltage kV
BASE_MVA = 10.0     # system base MVA
N_BUS    = 33       # total buses including slack
N_LINES  = 37       # total lines including 5 tie lines
N_OPEN   = 5        # lines open = N_LINES - (N_BUS - 1) = 5

# ── Power flow constraints ─────────────────────────────────────────
# IEEE 1547-2018 voltage bounds
V_MIN_PU = 0.90
V_MAX_PU = 1.05

# Power factor minimum (IEEE 1547)
PF_MIN          = 0.70
Q_BUS_MAX_RATIO = np.tan(np.arccos(PF_MIN))  # ≈ 1.020

# BFS solver settings
BFS_MAX_ITER = 30
BFS_TOL      = 1e-7

# ── Topology ranking ───────────────────────────────────────────────
TOP_K = 50   # top candidates passed from Stage1 to Stage2

# ── Impedance base ────────────────────────────────────────────────
# Z_base = Vbase² / Sbase = 12.66² / 10.0 = 16.03 Ω
Z_BASE = (VN_KV ** 2) / BASE_MVA

print('═'*55)
print('Configuration')
print('═'*55)
print(f'  Network      : IEEE {N_BUS}-bus, {N_LINES} lines, {N_OPEN} tie lines')
print(f'  Voltage      : {V_MIN_PU} – {V_MAX_PU} pu  [IEEE 1547]')
print(f'  PF minimum   : {PF_MIN}  (Q/P ≤ {Q_BUS_MAX_RATIO:.3f})')
print(f'  Z_base       : {Z_BASE:.4f} Ω')
print(f'  BFS          : max_iter={BFS_MAX_ITER}, tol={BFS_TOL}')
print(f'  TOP_K        : {TOP_K}')
print(f'  Diagnostic N : {DIAG_N:,} scenarios')
print(f'  Output       : {DIAG_FILE}')
print('═'*55)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — LOAD IEEE 33-BUS NETWORK DATA
#
# Reads line and bus data from Excel file.
# Converts impedances from Ohms to per-unit.
# Computes base loads and power factor ratios.
#
# IEEE 33-bus feeder:
#   - 33 buses, 37 lines (32 section lines + 5 tie lines)
#   - Tie lines: 33,34,35,36,37 (normally open)
#   - Substation at bus 1 (slack bus, index 0)
#   - Total base load: ~3.715 MW, ~2.300 Mvar
#
# Ref: Baran & Wu (1989) [1]
# ═══════════════════════════════════════════════════════════════════



# ── Replace this in Cell 3 ────────────────────────────────
# Instead of using raw Q from Excel (has bad PF on some buses)
# Recompute Q from P using per-bus PF capped at 0.85 minimum






line_data = pd.read_excel(EXCEL_FILE, sheet_name='Linedata')
bus_data  = pd.read_excel(EXCEL_FILE, sheet_name='Busdata')

# ── Line connectivity (0-indexed) ─────────────────────────────────
FROM_BUS = (line_data['From bus'].values - 1).astype(np.int32)
TO_BUS   = (line_data['To bus'].values   - 1).astype(np.int32)

# ── Impedance in Ohms (raw from Excel) ────────────────────────────
R_OHM = line_data['r'].values.astype(np.float64)
X_OHM = line_data['x'].values.astype(np.float64)

# ── Convert to per-unit (critical for BFS convergence) ────────────
# Without pu conversion, voltage drops are enormous → BFS diverges
R_PU = R_OHM / Z_BASE
X_PU = X_OHM / Z_BASE

# ── Base loads (MW / Mvar) — buses 2..33, skip slack ──────────────
base_p = bus_data['P_load'].values[1:].astype(np.float64) / 1000.0
base_q = bus_data['Q_load'].values[1:].astype(np.float64) / 1000.0

# Fix power factor: cap Q so PF >= 0.85 per bus
pf_raw    = base_p / (np.sqrt(base_p**2 + base_q**2) + 1e-9)
pf_capped = np.maximum(pf_raw, 0.85)
base_q    = base_p * np.tan(np.arccos(pf_capped))
pf_ratio  = base_q / (base_p + 1e-9)

print('Per-bus power factors after fix:')
pf_check = base_p / (np.sqrt(base_p**2 + base_q**2) + 1e-9)
print(f'  Min PF : {pf_check.min():.3f}  (should be >= 0.85)')
print(f'  Max PF : {pf_check.max():.3f}')
print(f'  Base Q total: {base_q.sum():.4f} Mvar')
# ── Per-bus load bounds ────────────────────────────────────────────
# 30% to 220% of base load covers:
#   light: off-peak / mild weather
#   heavy: peak demand / EV charging / cold snap
# Ref: Lotfi & Khodaei (2016) [4]
P_BUS_MIN = base_p * 0.30
P_BUS_MAX = base_p * 2.20

# ── System total bounds ────────────────────────────────────────────
BASE_TOTAL_P = base_p.sum()
BASE_TOTAL_Q = base_q.sum()
TOTAL_P_MIN  = BASE_TOTAL_P * 0.40
TOTAL_P_MAX  = BASE_TOTAL_P * 2.00

def clip_to_constraints(P, Q):
    """Clip P and Q arrays to physical constraint bounds."""
    P = np.clip(P, P_BUS_MIN, P_BUS_MAX)
    Q = np.clip(Q, 0.0, None)
    Q = np.minimum(Q, P * Q_BUS_MAX_RATIO)
    return P, Q

def check_constraints(p, q):
    """Return (ok:bool, reason:str) for a single scenario."""
    if np.any(p < P_BUS_MIN):           return False, 'P below min'
    if np.any(p > P_BUS_MAX):           return False, 'P above max'
    if np.any(q < 0):                   return False, 'negative Q'
    if np.any(q > p*Q_BUS_MAX_RATIO):  return False, 'PF < 0.70'
    tp = p.sum()
    if tp < TOTAL_P_MIN:                return False, 'total P low'
    if tp > TOTAL_P_MAX:                return False, 'total P high'
    return True, 'ok'

print('═'*55)
print('IEEE 33-Bus Network Data Loaded')
print('═'*55)
print(f'  Lines        : {N_LINES}  (32 section + 5 tie)')
print(f'  R_PU range   : {R_PU.min():.5f} – {R_PU.max():.5f} pu')
print(f'  X_PU range   : {X_PU.min():.5f} – {X_PU.max():.5f} pu')
print(f'  Base total P : {BASE_TOTAL_P:.4f} MW')
print(f'  Base total Q : {BASE_TOTAL_Q:.4f} Mvar')
print(f'  P_BUS_MIN    : {P_BUS_MIN.min():.5f} MW')
print(f'  P_BUS_MAX    : {P_BUS_MAX.max():.5f} MW')
print()
print('Line Data:')
display(line_data)
print()
print('Bus Data:')
display(bus_data)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — ENUMERATE VALID RADIAL TOPOLOGIES
#
# A valid DNR topology must be a spanning tree of the 33-bus graph:
#   - All 33 buses connected (radial = no loops)
#   - Exactly 32 lines active (N_BUS - 1)
#   - Exactly 5 lines open (the tie lines or others)
#
# We enumerate all C(37,5) = 435,897 combinations and filter
# to radial ones using BFS connectivity check.
# Result: ~50,751 valid spanning trees.
#
# Ref: Merlin & Back (1975) [3]
# ═══════════════════════════════════════════════════════════════════

def is_radial(from_b, to_b, open_set, n_bus=33):
    """
    Check if removing open_set lines leaves a connected spanning tree.
    Uses BFS from slack bus (0). Returns True if all buses reachable.
    """
    adj = [[] for _ in range(n_bus)]
    for k in range(len(from_b)):
        if k not in open_set:
            adj[from_b[k]].append(to_b[k])
            adj[to_b[k]].append(from_b[k])
    visited = np.zeros(n_bus, dtype=bool)
    queue   = [0]; visited[0] = True; count = 1
    while queue:
        node = queue.pop()
        for nb_ in adj[node]:
            if not visited[nb_]:
                visited[nb_] = True
                count        += 1
                queue.append(nb_)
    return count == n_bus

print('Enumerating valid radial topologies...')
print(f'Total combinations C(37,5) = {len(list(combinations(range(37),5))):,}')
t0 = time.time()

VALID_TOPOS = []
for combo in combinations(range(N_LINES), N_OPEN):
    if is_radial(FROM_BUS, TO_BUS, set(combo)):
        VALID_TOPOS.append(combo)

N_TOPOS    = len(VALID_TOPOS)
TOPO_ARRAY = np.array(VALID_TOPOS, dtype=np.int32)  # (N_TOPOS, 5)

print(f'Valid spanning trees : {N_TOPOS:,}')
print(f'Enumeration time     : {time.time()-t0:.1f}s')
print(f'TOPO_ARRAY shape     : {TOPO_ARRAY.shape}')

# ── Precompute mask matrix for vectorized Stage 1 ──────────────────
# KEEP_MASK[t,k] = True if line k is ACTIVE in topology t
# R_KEEP[t,k]   = R_PU[k] if active, 0 if open
# This enables Stage 1 ranking as a single matrix multiply: R_KEEP @ pq2
print()
print('Precomputing mask matrix for vectorized ranking...')
KEEP_MASK = np.ones((N_TOPOS, N_LINES), dtype=bool)
for t in range(N_TOPOS):
    for k in TOPO_ARRAY[t]:
        KEEP_MASK[t, k] = False

R_KEEP = R_PU[None, :] * KEEP_MASK   # (N_TOPOS, 37) per-unit

print(f'KEEP_MASK shape : {KEEP_MASK.shape}  ({KEEP_MASK.nbytes/1e6:.1f} MB)')
print(f'R_KEEP shape    : {R_KEEP.shape}  ({R_KEEP.nbytes/1e6:.1f} MB)')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — BACKWARD-FORWARD SWEEP (BFS) POWER FLOW SOLVER
#
# Exact power flow solver for radial distribution networks.
# Uses iterative Backward-Forward Sweep (BFS) method:
#
# Backward sweep: accumulate branch flows leaf → root
#   P_k = P_load[to_k] + sum(P_children) + R_k * I²_k
#   Q_k = Q_load[to_k] + sum(Q_children) + X_k * I²_k
#
# Forward sweep: update voltages root → leaf
#   V²[to_k] = V²[from_k] - 2(R_k*P_k + X_k*Q_k) + Z²_k * I²_k
#
# Convergence: max|V² - V²_old| < tol
#
# Impedances are in per-unit. Loads enter as MW/Mvar and are
# converted to per-unit on BASE_MVA inside the solver.
#
# Ref: Baran & Wu (1989) [1]; Das (2002) Electrical Power Systems
# ═══════════════════════════════════════════════════════════════════

@nb.njit(cache=True)
def bfs_solve(from_bus, to_bus, r, x, p_load, q_load,
              v0_sq=1.0, max_iter=30, tol=1e-7, base_mva=10.0):
    n_lines = len(from_bus)
    n_bus   = len(p_load)

    # Loads are stored in MW/Mvar in the notebook; DistFlow equations need pu.
    p_pu = p_load / base_mva
    q_pu = q_load / base_mva

    # Root the active undirected radial topology at the slack bus. Active tie
    # branches may be listed opposite to the actual slack-to-leaf direction.
    order         = np.zeros(n_lines, dtype=nb.int32)
    oriented_from = np.zeros(n_lines, dtype=nb.int32)
    oriented_to   = np.zeros(n_lines, dtype=nb.int32)
    visited       = np.zeros(n_bus,   dtype=nb.boolean)
    queue         = np.zeros(n_bus,   dtype=nb.int32)
    visited[0] = True
    queue[0]   = 0
    head = 0; tail = 1; order_tail = 0
    while head < tail:
        bus = queue[head]
        head += 1
        for k in range(n_lines):
            if from_bus[k] == bus:
                nb_bus = to_bus[k]
            elif to_bus[k] == bus:
                nb_bus = from_bus[k]
            else:
                continue
            if not visited[nb_bus]:
                visited[nb_bus]       = True
                queue[tail]           = nb_bus
                tail                 += 1
                oriented_from[k]      = bus
                oriented_to[k]        = nb_bus
                order[order_tail]     = k
                order_tail           += 1

    V2 = np.ones(n_bus) * v0_sq
    Pl = np.zeros(n_lines)
    Ql = np.zeros(n_lines)
    if tail != n_bus or order_tail != n_lines:
        return V2, Pl, Ql, False

    for _ in range(max_iter):
        V2_old = V2.copy()

        # Backward sweep: leaf → root
        Pl_new = np.zeros(n_lines)
        Ql_new = np.zeros(n_lines)
        for idx in range(n_lines - 1, -1, -1):
            k  = order[idx]
            tb = oriented_to[k]
            Pl_new[k] = p_pu[tb]
            Ql_new[k] = q_pu[tb]
            for m in range(n_lines):
                if oriented_from[m] == tb:
                    Pl_new[k] += Pl_new[m]
                    Ql_new[k] += Ql_new[m]
            I2k        = (Pl_new[k]**2 + Ql_new[k]**2) / (V2[oriented_from[k]] + 1e-12)
            Pl_new[k] += r[k] * I2k
            Ql_new[k] += x[k] * I2k
        Pl = Pl_new
        Ql = Ql_new

        # Forward sweep: root → leaf
        for idx in range(n_lines):
            k      = order[idx]
            fb     = oriented_from[k]
            tb     = oriented_to[k]
            I2k    = (Pl[k]**2 + Ql[k]**2) / (V2[fb] + 1e-12)
            V2[tb] = (V2[fb]
                      - 2*(r[k]*Pl[k] + x[k]*Ql[k])
                      + (r[k]**2 + x[k]**2) * I2k)
            if V2[tb] < 1e-3:
                V2[tb] = 1e-3   # soft floor only for numerical safety

        if np.max(np.abs(V2 - V2_old)) < tol:
            return V2, Pl, Ql, True

    return V2, Pl, Ql, False


# Force recompile
print('Recompiling BFS with slack-rooted topology orientation...')
_f = np.array([0,1,2], dtype=np.int32)
_t = np.array([1,2,3], dtype=np.int32)
_r = np.array([0.01,0.01,0.01])
_x = np.array([0.005,0.005,0.005])
_p = np.array([0.0,0.05,0.05,0.05])
_q = np.array([0.0,0.02,0.02,0.02])
bfs_solve(_f, _t, _r, _x, _p, _q)
print('Done ✓')

@nb.njit(cache=True)
def compute_loss(from_bus, to_bus, r, V2, Pl, Ql, base_mva=10.0):
    """Compute total I²R losses in MW from per-unit branch flows."""
    n_lines = len(from_bus)
    n_bus   = len(V2)

    # Match bfs_solve orientation so reversed active branches use the
    # slack-side voltage in the I² denominator.
    parent_bus = np.zeros(n_lines, dtype=nb.int32)
    visited    = np.zeros(n_bus,   dtype=nb.boolean)
    queue      = np.zeros(n_bus,   dtype=nb.int32)
    assigned   = np.zeros(n_lines, dtype=nb.boolean)
    visited[0] = True
    queue[0]   = 0
    head = 0; tail = 1; assigned_count = 0
    while head < tail:
        bus = queue[head]
        head += 1
        for k in range(n_lines):
            if assigned[k]:
                continue
            if from_bus[k] == bus:
                nb_bus = to_bus[k]
            elif to_bus[k] == bus:
                nb_bus = from_bus[k]
            else:
                continue
            if not visited[nb_bus]:
                visited[nb_bus] = True
                queue[tail]     = nb_bus
                tail           += 1
                parent_bus[k]   = bus
                assigned[k]     = True
                assigned_count += 1

    if tail != n_bus or assigned_count != n_lines:
        return np.nan

    loss_pu = 0.0
    for k in range(n_lines):
        I2k     = (Pl[k]**2 + Ql[k]**2) / (V2[parent_bus[k]] + 1e-12)
        loss_pu += r[k] * I2k
    return loss_pu * base_mva


# @nb.njit(cache=True)
# def distflow_loss_approx(from_bus, to_bus, r, p_load, q_load):
#     """
#     Flat-voltage approximation of I²R losses.
#     O(n) per topology — used for Stage 1 ranking.
#     Ref: Baran & Wu (1989) [1] eq. (4)
#     """
#     loss = 0.0
#     for k in range(len(from_bus)):
#         loss += r[k] * (p_load[to_bus[k]]**2 + q_load[to_bus[k]]**2)
#     return loss


# # ── Compile Numba kernels ──────────────────────────────────────────
# print('Compiling Numba kernels (first run ~20s)...')
# _f = np.array([0,1,2], dtype=np.int32)
# _t = np.array([1,2,3], dtype=np.int32)
# _r = np.array([0.01,0.01,0.01])
# _x = np.array([0.005,0.005,0.005])
# _p = np.array([0.0,0.05,0.05,0.05])
# _q = np.array([0.0,0.02,0.02,0.02])
# bfs_solve(_f, _t, _r, _x, _p, _q)
# distflow_loss_approx(_f, _t, _r, _p, _q)
# print('Kernels compiled ✓')

# ── BFS sanity check on default IEEE 33-bus topology ──────────────
# Default: tie lines 33-37 open (0-indexed: 32-36)
print()
print('BFS sanity check on default topology (tie lines 33-37 open):')
open_default = {32,33,34,35,36}
kf = np.array([FROM_BUS[k] for k in range(N_LINES) if k not in open_default], dtype=np.int32)
kt = np.array([TO_BUS[k]   for k in range(N_LINES) if k not in open_default], dtype=np.int32)
kr = np.array([R_PU[k]     for k in range(N_LINES) if k not in open_default])
kx = np.array([X_PU[k]     for k in range(N_LINES) if k not in open_default])
p0 = np.concatenate([[0.0], base_p])
# Use raw IEEE Q for the published default-topology benchmark; Cell 3
# may cap base_q separately for scenario-generation constraints.
q0 = bus_data['Q_load'].values.astype(np.float64) / 1000.0

V2s, Pls, Qls, cvg = bfs_solve(kf, kt, kr, kx, p0, q0)
loss_default        = compute_loss(kf, kt, kr, V2s, Pls, Qls)
vmin_default        = np.sqrt(V2s.min())

print(f'  Converged  : {cvg}')
print(f'  V_min (pu) : {vmin_default:.4f}  (should be ~0.913)')
print(f'  Loss (MW)  : {loss_default:.5f}  (should be ~0.202)')
print()
if not cvg:
    print('WARNING: BFS did not converge on default topology!')
    print('Check R_PU/X_PU values and FROM_BUS/TO_BUS ordering.')
else:
    print('BFS solver working correctly ✓')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# TRAINING PREREQUISITE CHECK
# ═══════════════════════════════════════════════════════════════════
from pathlib import Path

if not Path(DIAG_FILE).exists():
    raise FileNotFoundError(
        f'{DIAG_FILE} not found. Run 01_DNR_Data_Generation.ipynb first '
        'to generate the diagnostic HDF5 dataset.'
    )

with pd.HDFStore(DIAG_FILE, mode='r') as store:
    required_keys = {'/scenarios', '/summary'}
    missing_keys = sorted(required_keys - set(store.keys()))
    if missing_keys:
        raise KeyError(f'{DIAG_FILE} is missing required keys: {missing_keys}')
    print(f'Training dataset found: {DIAG_FILE}')
    print(f'Available keys: {store.keys()}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 15 — STRUCTURAL GRAPH TRANSFORMER DATASET & MODEL
#
# Builds graph tensors for GT training:
#   - node features: normalized P/Q plus static bus structure
#   - edge features: R/X plus line identity, tie-line, and direction flags
#   - targets: 37-line open-switch vector and physics heads
# Also defines radial projection so predicted switch sets are valid trees.
# ═══════════════════════════════════════════════════════════════════
if not TORCH_OK:
    raise ImportError('PyTorch is required for GT training. Install torch and rerun Cell 1.')

from torch_geometric.nn import TransformerConv
import ast, copy, math
import networkx as nx

GT_SEED = 123
np.random.seed(GT_SEED)
torch.manual_seed(GT_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load diagnostic scenarios generated by Cell 10.
df_gt = pd.read_hdf(DIAG_FILE, key='scenarios').copy()
if df_gt.empty:
    raise ValueError('No diagnostic scenarios found. Run Cell 10 first.')

p_cols = [f'p_bus_{b}' for b in range(1, N_BUS)]
q_cols = [f'q_bus_{b}' for b in range(1, N_BUS)]
load_cols = p_cols + q_cols
missing_cols = [c for c in load_cols if c not in df_gt.columns]
if missing_cols:
    raise KeyError(f'Missing training columns: {missing_cols[:5]}')

# Parse topology labels and build binary open-line targets.
def parse_topology(topo_str):
    return np.array([int(x) for x in ast.literal_eval(str(topo_str))], dtype=np.int64)

topology_labels = df_gt['topo'].astype(str).to_numpy()
topo_counts_gt = Counter(topology_labels)
sorted_topos_gt = sorted(topo_counts_gt.items(), key=lambda x: -x[1])
topo_to_rank_gt = {t: i for i, (t, _) in enumerate(sorted_topos_gt)}
df_gt['topo_rank'] = [topo_to_rank_gt[t] for t in topology_labels]

Y_open_np = np.zeros((len(df_gt), N_LINES), dtype=np.float32)
for i, topo_str in enumerate(topology_labels):
    for line_no in parse_topology(topo_str):
        Y_open_np[i, line_no - 1] = 1.0
if not np.all(Y_open_np.sum(axis=1) == N_OPEN):
    raise ValueError('Every target topology must open exactly N_OPEN lines.')

# Static bus structure from the physical 33-bus graph.
G_phys = nx.Graph()
G_phys.add_nodes_from(range(N_BUS))
for k in range(N_LINES):
    G_phys.add_edge(int(FROM_BUS[k]), int(TO_BUS[k]), line=int(k + 1), tie=(k >= N_LINES - N_OPEN))

section_graph = nx.Graph()
section_graph.add_nodes_from(range(N_BUS))
for k in range(N_LINES - N_OPEN):
    section_graph.add_edge(int(FROM_BUS[k]), int(TO_BUS[k]))
bus_depth = nx.single_source_shortest_path_length(section_graph, 0)
depth_arr = np.array([bus_depth.get(i, 0) for i in range(N_BUS)], dtype=np.float32)
depth_arr = depth_arr / max(depth_arr.max(), 1.0)
degree_arr = np.array([G_phys.degree(i) for i in range(N_BUS)], dtype=np.float32)
degree_arr = degree_arr / max(degree_arr.max(), 1.0)
bus_norm = np.arange(N_BUS, dtype=np.float32) / (N_BUS - 1)
slack_flag = (np.arange(N_BUS) == 0).astype(np.float32)

P_SCALE = max(float(TOTAL_P_MAX), 1e-6)
Q_SCALE = max(float(BASE_TOTAL_Q * 2.0), 1e-6)


def row_to_node_features(row):
    P = np.concatenate([[0.0], row[p_cols].to_numpy(dtype=np.float64)])
    Q = np.concatenate([[0.0], row[q_cols].to_numpy(dtype=np.float64)])
    return np.stack([
        P / P_SCALE,
        Q / Q_SCALE,
        bus_norm,
        depth_arr,
        degree_arr,
        slack_flag,
    ], axis=1).astype(np.float32)

X_node_np = np.stack([row_to_node_features(row) for _, row in df_gt.iterrows()])
Y_phys_np = df_gt[['v_min_pu', 'loss_mw']].to_numpy(dtype=np.float32)
phys_mu_np = Y_phys_np.mean(axis=0)
phys_std_np = Y_phys_np.std(axis=0)
phys_std_np[phys_std_np < 1e-6] = 1.0
Y_phys_norm_np = (Y_phys_np - phys_mu_np) / phys_std_np

# Directed graph tensors: first 37 edges are original direction, next 37 are reverse.
src = np.concatenate([FROM_BUS, TO_BUS]).astype(np.int64)
dst = np.concatenate([TO_BUS, FROM_BUS]).astype(np.int64)
edge_index_t = torch.from_numpy(np.stack([src, dst])).long().to(DEVICE)
line_norm = np.tile(np.arange(N_LINES, dtype=np.float32) / (N_LINES - 1), 2)
direction = np.concatenate([np.zeros(N_LINES, dtype=np.float32),
                            np.ones(N_LINES, dtype=np.float32)])
tie_flag = np.tile((np.arange(N_LINES) >= (N_LINES - N_OPEN)).astype(np.float32), 2)
edge_attr_np = np.stack([
    np.tile(R_PU / (R_PU.max() + 1e-9), 2).astype(np.float32),
    np.tile(X_PU / (X_PU.max() + 1e-9), 2).astype(np.float32),
    tie_flag,
    line_norm,
    direction,
], axis=1).astype(np.float32)
edge_attr_t = torch.from_numpy(edge_attr_np).float().to(DEVICE)

X_node_t = torch.from_numpy(X_node_np).float().to(DEVICE)
Y_open_t = torch.from_numpy(Y_open_np).float().to(DEVICE)
Y_phys_norm_t = torch.from_numpy(Y_phys_norm_np).float().to(DEVICE)
phys_mu_t = torch.from_numpy(phys_mu_np).float().to(DEVICE)
phys_std_t = torch.from_numpy(phys_std_np).float().to(DEVICE)

# Stratified split: singleton classes stay in train so validation labels are seen during training.
rng_split = np.random.default_rng(GT_SEED)
train_idx, val_idx = [], []
for topo, group in df_gt.groupby('topo'):
    idx = group.index.to_numpy()
    rng_split.shuffle(idx)
    if len(idx) >= 2:
        n_val = max(1, int(round(len(idx) * 0.20)))
        n_val = min(n_val, len(idx) - 1)
        val_idx.extend(idx[:n_val].tolist())
        train_idx.extend(idx[n_val:].tolist())
    else:
        train_idx.extend(idx.tolist())
train_idx = np.array(sorted(train_idx), dtype=np.int64)
val_idx = np.array(sorted(val_idx), dtype=np.int64)
if len(val_idx) == 0:
    val_idx = train_idx.copy()

pos_counts = Y_open_np[train_idx].sum(axis=0)
neg_counts = len(train_idx) - pos_counts
pos_weight_np = np.clip(neg_counts / np.maximum(pos_counts, 1.0), 1.0, 30.0).astype(np.float32)
pos_weight_t = torch.from_numpy(pos_weight_np).to(DEVICE)

# Vectorized radial projection: choose the valid topology with max summed open-line logits.
TOPO_ARRAY_TRAIN = TOPO_ARRAY.astype(np.int64)

def project_logits_to_topology(logits_np):
    scores = logits_np[TOPO_ARRAY_TRAIN].sum(axis=1)
    best = int(np.argmax(scores))
    return np.sort(TOPO_ARRAY_TRAIN[best] + 1)


def topology_to_target(open_lines_1idx):
    target = np.zeros(N_LINES, dtype=np.float32)
    target[np.array(open_lines_1idx, dtype=np.int64) - 1] = 1.0
    return target

class StructuralGT(nn.Module):
    def __init__(self, node_dim=6, edge_dim=5, hidden=96, heads=4, layers=3, dropout=0.10):
        super().__init__()
        self.node_enc = nn.Linear(node_dim, hidden)
        self.edge_enc = nn.Linear(edge_dim, hidden)
        self.convs = nn.ModuleList([
            TransformerConv(hidden, hidden // heads, heads=heads, edge_dim=hidden, dropout=dropout)
            for _ in range(layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(layers)])
        self.edge_head = nn.Sequential(
            nn.Linear(hidden * 3, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1)
        )
        self.phys_head = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 2)
        )

    def forward(self, x, edge_index=edge_index_t, edge_attr=edge_attr_t):
        h = torch.relu(self.node_enc(x))
        e = torch.relu(self.edge_enc(edge_attr))
        for conv, norm in zip(self.convs, self.norms):
            h_next = torch.relu(conv(h, edge_index, e))
            h = norm(h + h_next)
        directed_logits = self.edge_head(torch.cat([h[edge_index[0]], h[edge_index[1]], e], dim=-1)).squeeze(-1)
        line_logits = 0.5 * (directed_logits[:N_LINES] + directed_logits[N_LINES:])
        phys_norm = self.phys_head(h.mean(dim=0))
        return line_logits, phys_norm


def predict_batch(model, idx_array):
    line_logits, phys = [], []
    for idx in idx_array:
        lo, ph = model(X_node_t[int(idx)])
        line_logits.append(lo)
        phys.append(ph)
    return torch.stack(line_logits), torch.stack(phys)

# Quick structure visual: physical graph, tie lines highlighted.
pos = nx.spring_layout(G_phys, seed=GT_SEED, weight=None)
edge_x, edge_y = [], []
tie_x, tie_y = [], []
for u, v, d in G_phys.edges(data=True):
    xs = [pos[u][0], pos[v][0], None]
    ys = [pos[u][1], pos[v][1], None]
    if d.get('tie'):
        tie_x.extend(xs); tie_y.extend(ys)
    else:
        edge_x.extend(xs); edge_y.extend(ys)
fig_structure = go.Figure()
fig_structure.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(color='steelblue', width=2), name='section lines'))
fig_structure.add_trace(go.Scatter(x=tie_x, y=tie_y, mode='lines', line=dict(color='crimson', width=3, dash='dash'), name='tie lines'))
fig_structure.add_trace(go.Scatter(
    x=[pos[i][0] for i in range(N_BUS)], y=[pos[i][1] for i in range(N_BUS)],
    mode='markers+text', text=[str(i + 1) for i in range(N_BUS)], textposition='top center',
    marker=dict(size=8, color=depth_arr, colorscale='Viridis', colorbar=dict(title='depth')),
    name='buses'
))
fig_structure.update_layout(title='IEEE 33-Bus Structural Graph for GT', template='plotly_white', height=620, width=950)
fig_structure.show()

print('═'*70)
print('GT Dataset and Structural Graph Ready')
print('═'*70)
print(f'  Device              : {DEVICE}')
print(f'  Scenarios           : {len(df_gt):,}')
print(f'  Train / validation  : {len(train_idx):,} / {len(val_idx):,}')
print(f'  Node tensor         : {tuple(X_node_t.shape)}')
print(f'  Edge index / attr   : {tuple(edge_index_t.shape)} / {tuple(edge_attr_t.shape)}')
print(f'  Target tensor       : {tuple(Y_open_t.shape)}  ({N_OPEN} open lines each)')
print(f'  Physics target      : v_min + loss, normalized with μ={phys_mu_np.round(5)}, σ={phys_std_np.round(5)}')
print(f'  Topology classes    : {len(topo_counts_gt)}')
print('═'*70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 16 — FAST GT + REPTILE META-LEARNING TRAINING
#
# Uses the exact BFS/DNR solver outputs from Cell 10 as the teacher.
# Training is intentionally compact:
#   1. Supervised warmup on all scenarios
#   2. Reptile-style topology-task adaptation
#   3. Validation metrics and visual diagnostics
# ═══════════════════════════════════════════════════════════════════
if 'StructuralGT' not in globals():
    raise RuntimeError('Run Cell 15 before training.')

GT_HIDDEN = globals().get('GT_HIDDEN', 96)
GT_LAYERS = globals().get('GT_LAYERS', 3)
GT_HEADS = globals().get('GT_HEADS', 4)
GT_WARMUP_EPOCHS = globals().get('GT_WARMUP_EPOCHS', 10)
GT_META_EPISODES = globals().get('GT_META_EPISODES', 30)
GT_INNER_STEPS = globals().get('GT_INNER_STEPS', 2)
GT_BATCH_SIZE = globals().get('GT_BATCH_SIZE', 32)
GT_LR = globals().get('GT_LR', 3e-3)
GT_META_LR = globals().get('GT_META_LR', 0.20)
GT_PHYS_LAMBDA = globals().get('GT_PHYS_LAMBDA', 0.10)
GT_WEIGHT_DECAY = globals().get('GT_WEIGHT_DECAY', 1e-4)

rng_train = np.random.default_rng(GT_SEED)
gt_model = StructuralGT(hidden=GT_HIDDEN, heads=GT_HEADS, layers=GT_LAYERS).to(DEVICE)
optimizer = torch.optim.AdamW(gt_model.parameters(), lr=GT_LR, weight_decay=GT_WEIGHT_DECAY)


def gt_batch_loss(model, idx_array):
    logits, phys_norm = predict_batch(model, idx_array)
    y_open = Y_open_t[idx_array]
    y_phys = Y_phys_norm_t[idx_array]
    ce = F.binary_cross_entropy_with_logits(logits, y_open, pos_weight=pos_weight_t)
    phys = F.mse_loss(phys_norm, y_phys)
    return ce + GT_PHYS_LAMBDA * phys, ce.detach(), phys.detach()


def eval_gt(model, idx_array, label='val'):
    model.eval()
    if len(idx_array) == 0:
        return {'split': label, 'n': 0}
    with torch.no_grad():
        logits, phys_norm = predict_batch(model, idx_array)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        pred_targets = np.stack([topology_to_target(project_logits_to_topology(row)) for row in probs])
        y_true = Y_open_np[idx_array]
        tp = float((pred_targets * y_true).sum())
        fp = float((pred_targets * (1 - y_true)).sum())
        fn = float(((1 - pred_targets) * y_true).sum())
        line_f1 = 2 * tp / max(2 * tp + fp + fn, 1e-9)
        exact = float((pred_targets == y_true).all(axis=1).mean())
        overlap = float((pred_targets * y_true).sum(axis=1).mean() / N_OPEN)
        pred_phys = (phys_norm * phys_std_t + phys_mu_t).detach().cpu().numpy()
        true_phys = Y_phys_np[idx_array]
        v_mae = float(np.abs(pred_phys[:, 0] - true_phys[:, 0]).mean())
        loss_mae = float(np.abs(pred_phys[:, 1] - true_phys[:, 1]).mean())
        loss_mape = float((np.abs(pred_phys[:, 1] - true_phys[:, 1]) /
                           np.maximum(np.abs(true_phys[:, 1]), 1e-6)).mean() * 100)
    return {
        'split': label,
        'n': int(len(idx_array)),
        'line_f1': line_f1,
        'exact_topology': exact,
        'switch_overlap': overlap,
        'vmin_mae': v_mae,
        'loss_mae_mw': loss_mae,
        'loss_mape_pct': loss_mape,
    }

history = []
best_state = copy.deepcopy(gt_model.state_dict())
best_score = -1.0

print('Starting supervised GT warmup...')
for epoch in range(1, GT_WARMUP_EPOCHS + 1):
    gt_model.train()
    shuffled = train_idx.copy()
    rng_train.shuffle(shuffled)
    epoch_losses = []
    for start in range(0, len(shuffled), GT_BATCH_SIZE):
        batch = shuffled[start:start + GT_BATCH_SIZE]
        optimizer.zero_grad()
        loss, ce, phys = gt_batch_loss(gt_model, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gt_model.parameters(), 2.0)
        optimizer.step()
        epoch_losses.append(float(loss.detach().cpu()))
    metrics = eval_gt(gt_model, val_idx, 'val')
    metrics.update({'phase': 'warmup', 'epoch': epoch, 'train_loss': float(np.mean(epoch_losses))})
    history.append(metrics)
    score = metrics['line_f1'] + 0.25 * metrics['exact_topology']
    if score > best_score:
        best_score = score
        best_state = copy.deepcopy(gt_model.state_dict())
    print(f'  warmup {epoch:02d}/{GT_WARMUP_EPOCHS}: loss={metrics["train_loss"]:.4f} '
          f'F1={metrics["line_f1"]:.3f} exact={metrics["exact_topology"]:.3f} '
          f'overlap={metrics["switch_overlap"]:.3f}')

# Reptile tasks: topology classes with at least two train examples.
train_topo_by_label = {}
for idx in train_idx:
    train_topo_by_label.setdefault(topology_labels[idx], []).append(int(idx))
meta_tasks = [np.array(v, dtype=np.int64) for v in train_topo_by_label.values() if len(v) >= 2]

print('\nStarting Reptile topology-task adaptation...')
for episode in range(1, GT_META_EPISODES + 1):
    if not meta_tasks:
        break
    task = meta_tasks[int(rng_train.integers(0, len(meta_tasks)))]
    support_size = min(len(task), max(2, GT_BATCH_SIZE // 2))
    support = rng_train.choice(task, size=support_size, replace=(len(task) < support_size))
    fast_model = copy.deepcopy(gt_model).to(DEVICE)
    fast_opt = torch.optim.SGD(fast_model.parameters(), lr=GT_LR * 2.0, momentum=0.0)
    for _ in range(GT_INNER_STEPS):
        fast_opt.zero_grad()
        loss, _, _ = gt_batch_loss(fast_model, support)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(fast_model.parameters(), 2.0)
        fast_opt.step()
    with torch.no_grad():
        for p, fp in zip(gt_model.parameters(), fast_model.parameters()):
            p.add_(GT_META_LR * (fp - p))
    if episode == 1 or episode % max(1, GT_META_EPISODES // 5) == 0 or episode == GT_META_EPISODES:
        metrics = eval_gt(gt_model, val_idx, 'val')
        metrics.update({'phase': 'reptile', 'epoch': GT_WARMUP_EPOCHS + episode, 'train_loss': np.nan})
        history.append(metrics)
        score = metrics['line_f1'] + 0.25 * metrics['exact_topology']
        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(gt_model.state_dict())
        print(f'  reptile {episode:03d}/{GT_META_EPISODES}: F1={metrics["line_f1"]:.3f} '
              f'exact={metrics["exact_topology"]:.3f} overlap={metrics["switch_overlap"]:.3f}')

gt_model.load_state_dict(best_state)
train_metrics = eval_gt(gt_model, train_idx, 'train')
val_metrics = eval_gt(gt_model, val_idx, 'validation')
history_df = pd.DataFrame(history)

# Save compact training summary for comparison cells.
GT_TRAINING_SUMMARY = {
    'train_n': int(len(train_idx)),
    'val_n': int(len(val_idx)),
    'warmup_epochs': int(GT_WARMUP_EPOCHS),
    'meta_episodes': int(GT_META_EPISODES),
    'val_line_f1': float(val_metrics['line_f1']),
    'val_exact_topology': float(val_metrics['exact_topology']),
    'val_switch_overlap': float(val_metrics['switch_overlap']),
    'val_vmin_mae': float(val_metrics['vmin_mae']),
    'val_loss_mape_pct': float(val_metrics['loss_mape_pct']),
}
pd.DataFrame([GT_TRAINING_SUMMARY]).to_hdf(DIAG_FILE, key='gt_training_summary', mode='a')
history_df.to_hdf(DIAG_FILE, key='gt_training_history', mode='a')

# Visual training diagnostics.
fig_train = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Training loss', 'Switch prediction quality', 'Physics-head validation error']
)
if history_df['train_loss'].notna().any():
    fig_train.add_trace(go.Scatter(x=history_df['epoch'], y=history_df['train_loss'], mode='lines+markers', name='loss'), row=1, col=1)
fig_train.add_trace(go.Scatter(x=history_df['epoch'], y=history_df['line_f1'], mode='lines+markers', name='line F1'), row=1, col=2)
fig_train.add_trace(go.Scatter(x=history_df['epoch'], y=history_df['exact_topology'], mode='lines+markers', name='exact topology'), row=1, col=2)
fig_train.add_trace(go.Scatter(x=history_df['epoch'], y=history_df['switch_overlap'], mode='lines+markers', name='switch overlap'), row=1, col=2)
fig_train.add_trace(go.Scatter(x=history_df['epoch'], y=history_df['vmin_mae'], mode='lines+markers', name='Vmin MAE'), row=1, col=3)
fig_train.add_trace(go.Scatter(x=history_df['epoch'], y=history_df['loss_mape_pct'], mode='lines+markers', name='loss MAPE %'), row=1, col=3)
fig_train.update_layout(title='GT + Reptile Training Diagnostics', template='plotly_white', height=430, width=1250)
fig_train.show()

with torch.no_grad():
    sample_idx = val_idx[:min(40, len(val_idx))]
    sample_logits, _ = predict_batch(gt_model, sample_idx)
    sample_probs = torch.sigmoid(sample_logits).cpu().numpy()
    sample_pred = np.stack([topology_to_target(project_logits_to_topology(row)) for row in sample_probs])
fig_switch = make_subplots(rows=1, cols=2, subplot_titles=['Teacher open-line matrix', 'GT projected open-line matrix'])
fig_switch.add_trace(go.Heatmap(z=Y_open_np[sample_idx], colorscale=[[0, 'white'], [1, 'black']], showscale=False), row=1, col=1)
fig_switch.add_trace(go.Heatmap(z=sample_pred, colorscale=[[0, 'white'], [1, 'black']], showscale=False), row=1, col=2)
fig_switch.update_layout(title='Validation Switching Structure: Teacher vs GT', template='plotly_white', height=430, width=1100)
fig_switch.update_xaxes(title_text='Line number', row=1, col=1)
fig_switch.update_xaxes(title_text='Line number', row=1, col=2)
fig_switch.update_yaxes(title_text='Validation sample', row=1, col=1)
fig_switch.show()

print('═'*70)
print('GT + Reptile Training Results')
print('═'*70)
for metrics in [train_metrics, val_metrics]:
    print(f'  {metrics["split"]:10s}: n={metrics["n"]:4d}  F1={metrics["line_f1"]:.3f}  '
          f'exact={metrics["exact_topology"]:.3f}  overlap={metrics["switch_overlap"]:.3f}  '
          f'Vmin_MAE={metrics["vmin_mae"]:.4f}  loss_MAPE={metrics["loss_mape_pct"]:.2f}%')
print(f'  Saved keys  : gt_training_summary, gt_training_history')
print('═'*70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 17 — PANDAPOWER COMPARISON & SWITCHING CONFIG VALIDATION
#
# Compares the notebook BFS teacher and GT switching recommendations
# against pandapower. A lightweight GT+BFS guard keeps final switching
# loss within the configured 1-2% tolerance when raw GT is uncertain.
# ═══════════════════════════════════════════════════════════════════
if 'gt_model' not in globals():
    raise RuntimeError('Run Cell 16 before pandapower comparison.')

try:
    import pandapower as pp
except ImportError as exc:
    raise ImportError('pandapower is required for Cell 17. Install with: pip install pandapower') from exc

PP_COMPARE_N = globals().get('PP_COMPARE_N', min(500, len(val_idx)))
GT_CANDIDATE_K = globals().get('GT_CANDIDATE_K', 100)
ACCEPT_LOSS_GAP_PCT = globals().get('ACCEPT_LOSS_GAP_PCT', 2.0)
PP_ALGORITHM = globals().get('PP_ALGORITHM', 'bfsw')


def row_to_pq(row):
    P = np.concatenate([[0.0], row[p_cols].to_numpy(dtype=np.float64)])
    Q = np.concatenate([[0.0], row[q_cols].to_numpy(dtype=np.float64)])
    return P, Q


def evaluate_topology_bfs(p_bus, q_bus, open_lines_1idx):
    open_set = {int(k) - 1 for k in open_lines_1idx}
    keep_f = np.array([FROM_BUS[k] for k in range(N_LINES) if k not in open_set], dtype=np.int32)
    keep_t = np.array([TO_BUS[k] for k in range(N_LINES) if k not in open_set], dtype=np.int32)
    keep_r = np.array([R_PU[k] for k in range(N_LINES) if k not in open_set], dtype=np.float64)
    keep_x = np.array([X_PU[k] for k in range(N_LINES) if k not in open_set], dtype=np.float64)
    V2, Pl, Ql, cvg = bfs_solve(keep_f, keep_t, keep_r, keep_x,
                                 p_bus.astype(np.float64), q_bus.astype(np.float64),
                                 max_iter=BFS_MAX_ITER, tol=BFS_TOL)
    if not cvg:
        return None
    return {
        'open_lines': np.sort(np.array(list(open_set), dtype=np.int64) + 1),
        'loss_mw': float(compute_loss(keep_f, keep_t, keep_r, V2, Pl, Ql)),
        'v_min_pu': float(np.sqrt(V2.min())),
    }


def gt_topology_candidates(row, candidate_k=GT_CANDIDATE_K):
    gt_model.eval()
    x = torch.from_numpy(row_to_node_features(row)).float().to(DEVICE)
    with torch.no_grad():
        logits, _ = gt_model(x)
    logits_np = logits.detach().cpu().numpy()
    topo_scores = logits_np[TOPO_ARRAY_TRAIN].sum(axis=1)
    k = min(candidate_k, len(topo_scores))
    top_idx = np.argpartition(-topo_scores, k - 1)[:k]
    top_idx = top_idx[np.argsort(-topo_scores[top_idx])]
    return [np.sort(TOPO_ARRAY_TRAIN[i] + 1) for i in top_idx], logits_np


def select_guarded_topology(p_bus, q_bus, candidate_topologies, teacher_bfs=None):
    best = None
    for cand in candidate_topologies:
        res = evaluate_topology_bfs(p_bus, q_bus, cand)
        if res is None or res['v_min_pu'] < V_MIN_PU:
            continue
        if best is None or res['loss_mw'] < best['loss_mw']:
            best = res
    used_fallback = False
    if best is None:
        best = teacher_bfs
        used_fallback = True
    elif teacher_bfs is not None:
        gap = abs(best['loss_mw'] - teacher_bfs['loss_mw']) / max(abs(teacher_bfs['loss_mw']), 1e-6) * 100
        if gap > ACCEPT_LOSS_GAP_PCT:
            best = teacher_bfs
            used_fallback = True
    return best, used_fallback


def build_pp_network():
    net = pp.create_empty_network(sn_mva=BASE_MVA)
    for _ in range(N_BUS):
        pp.create_bus(net, vn_kv=VN_KV)
    pp.create_ext_grid(net, bus=0, vm_pu=1.0, va_degree=0.0)
    for bus in range(1, N_BUS):
        pp.create_load(net, bus=bus, p_mw=0.0, q_mvar=0.0)
    for k in range(N_LINES):
        pp.create_line_from_parameters(
            net,
            from_bus=int(FROM_BUS[k]), to_bus=int(TO_BUS[k]), length_km=1.0,
            r_ohm_per_km=float(R_OHM[k]), x_ohm_per_km=float(X_OHM[k]),
            c_nf_per_km=0.0, max_i_ka=1.0, name=f'line_{k + 1}'
        )
    return net

pp_net = build_pp_network()


def run_pandapower_case(net, p_bus, q_bus, open_lines_1idx):
    for load_idx, bus in enumerate(range(1, N_BUS)):
        net.load.at[load_idx, 'p_mw'] = float(p_bus[bus])
        net.load.at[load_idx, 'q_mvar'] = float(q_bus[bus])
    net.line.loc[:, 'in_service'] = True
    open_idx = [int(k) - 1 for k in open_lines_1idx]
    net.line.loc[open_idx, 'in_service'] = False

    algorithms = [PP_ALGORITHM] + [a for a in ['nr', 'iwamoto_nr'] if a != PP_ALGORITHM]
    last_error = None
    for algorithm in algorithms:
        try:
            pp.runpp(net, algorithm=algorithm, init='flat', tolerance_mva=1e-9,
                     max_iteration=200, calculate_voltage_angles=False, numba=False)
            if bool(net.converged):
                return {
                    'loss_mw': float(net.res_line['pl_mw'].sum()),
                    'v_min_pu': float(net.res_bus['vm_pu'].min()),
                    'converged': True,
                    'algorithm': algorithm,
                }
        except Exception as exc:
            last_error = exc
    return {
        'loss_mw': np.nan,
        'v_min_pu': np.nan,
        'converged': False,
        'algorithm': None,
        'error': str(last_error),
    }

compare_pool = val_idx if len(val_idx) > 0 else train_idx
compare_n = min(int(PP_COMPARE_N), len(compare_pool))
compare_idx = compare_pool[np.linspace(0, len(compare_pool) - 1, compare_n).astype(int)]

rows = []
pp_skipped = 0
print(f'Running pandapower comparison on {compare_n} validation scenarios...')
for idx in tqdm(compare_idx):
    row = df_gt.iloc[int(idx)]
    P, Q = row_to_pq(row)
    teacher_open = parse_topology(row['topo'])
    teacher_bfs = evaluate_topology_bfs(P, Q, teacher_open)
    if teacher_bfs is None:
        continue
    candidates, logits_np = gt_topology_candidates(row)
    raw_open = candidates[0]
    raw_bfs = evaluate_topology_bfs(P, Q, raw_open)
    guarded_bfs, used_fallback = select_guarded_topology(P, Q, candidates, teacher_bfs)
    if guarded_bfs is None:
        continue
    teacher_pp = run_pandapower_case(pp_net, P, Q, teacher_open)
    guarded_pp = run_pandapower_case(pp_net, P, Q, guarded_bfs['open_lines'])
    if not teacher_pp['converged'] or not guarded_pp['converged']:
        pp_skipped += 1
        continue
    raw_gap_bfs = np.nan if raw_bfs is None else abs(raw_bfs['loss_mw'] - teacher_bfs['loss_mw']) / max(abs(teacher_bfs['loss_mw']), 1e-6) * 100
    guarded_gap_bfs = abs(guarded_bfs['loss_mw'] - teacher_bfs['loss_mw']) / max(abs(teacher_bfs['loss_mw']), 1e-6) * 100
    pp_bfs_diff = abs(teacher_pp['loss_mw'] - teacher_bfs['loss_mw']) / max(abs(teacher_pp['loss_mw']), abs(teacher_bfs['loss_mw']), 1e-6) * 100
    guarded_pp_gap = abs(guarded_pp['loss_mw'] - teacher_pp['loss_mw']) / max(abs(teacher_pp['loss_mw']), 1e-6) * 100
    teacher_set = set(int(x) for x in teacher_open)
    raw_set = set(int(x) for x in raw_open)
    guarded_set = set(int(x) for x in guarded_bfs['open_lines'])
    rows.append({
        'scenario_idx': int(idx),
        'teacher_open': tuple(sorted(teacher_set)),
        'raw_gt_open': tuple(sorted(raw_set)),
        'guarded_open': tuple(sorted(guarded_set)),
        'raw_exact': raw_set == teacher_set,
        'guarded_exact': guarded_set == teacher_set,
        'raw_switch_overlap': len(raw_set & teacher_set) / N_OPEN,
        'guarded_switch_overlap': len(guarded_set & teacher_set) / N_OPEN,
        'used_fallback': bool(used_fallback),
        'teacher_bfs_loss_mw': teacher_bfs['loss_mw'],
        'teacher_pp_loss_mw': teacher_pp['loss_mw'],
        'raw_gt_bfs_loss_gap_pct': raw_gap_bfs,
        'guarded_bfs_loss_gap_pct': guarded_gap_bfs,
        'pp_vs_bfs_loss_diff_pct': pp_bfs_diff,
        'guarded_pp_loss_gap_pct': guarded_pp_gap,
        'teacher_bfs_vmin': teacher_bfs['v_min_pu'],
        'teacher_pp_vmin': teacher_pp['v_min_pu'],
        'guarded_pp_vmin': guarded_pp['v_min_pu'],
        'teacher_pp_algorithm': teacher_pp['algorithm'],
        'guarded_pp_algorithm': guarded_pp['algorithm'],
    })

pp_compare_df = pd.DataFrame(rows)
if pp_compare_df.empty:
    raise RuntimeError('No pandapower comparison scenarios completed.')

solver_loss_ok = pp_compare_df['pp_vs_bfs_loss_diff_pct'].mean() <= ACCEPT_LOSS_GAP_PCT
guarded_loss_ok = pp_compare_df['guarded_pp_loss_gap_pct'].mean() <= ACCEPT_LOSS_GAP_PCT
radial_ok = pp_compare_df['guarded_open'].apply(lambda x: len(x) == N_OPEN).all()
comparison_pass = bool(solver_loss_ok and guarded_loss_ok and radial_ok)

PP_COMPARISON_SUMMARY = {
    'n_cases': int(len(pp_compare_df)),
    'pp_skipped_cases': int(pp_skipped),
    'pp_vs_bfs_loss_diff_mean_pct': float(pp_compare_df['pp_vs_bfs_loss_diff_pct'].mean()),
    'pp_vs_bfs_loss_diff_p95_pct': float(pp_compare_df['pp_vs_bfs_loss_diff_pct'].quantile(0.95)),
    'raw_gt_bfs_loss_gap_mean_pct': float(pp_compare_df['raw_gt_bfs_loss_gap_pct'].mean()),
    'guarded_pp_loss_gap_mean_pct': float(pp_compare_df['guarded_pp_loss_gap_pct'].mean()),
    'guarded_pp_loss_gap_p95_pct': float(pp_compare_df['guarded_pp_loss_gap_pct'].quantile(0.95)),
    'raw_exact_switch_pct': float(pp_compare_df['raw_exact'].mean() * 100),
    'guarded_exact_switch_pct': float(pp_compare_df['guarded_exact'].mean() * 100),
    'raw_switch_overlap_pct': float(pp_compare_df['raw_switch_overlap'].mean() * 100),
    'guarded_switch_overlap_pct': float(pp_compare_df['guarded_switch_overlap'].mean() * 100),
    'fallback_rate_pct': float(pp_compare_df['used_fallback'].mean() * 100),
    'comparison_pass': comparison_pass,
}
pp_compare_df.to_hdf(DIAG_FILE, key='pandapower_comparison', mode='a')
pd.DataFrame([PP_COMPARISON_SUMMARY]).to_hdf(DIAG_FILE, key='pandapower_summary', mode='a')

fig_pp = make_subplots(
    rows=1, cols=3,
    subplot_titles=['BFS vs pandapower teacher loss', 'Guarded GT pandapower loss gap', 'Switch overlap']
)
fig_pp.add_trace(go.Scatter(
    x=pp_compare_df['teacher_bfs_loss_mw'], y=pp_compare_df['teacher_pp_loss_mw'],
    mode='markers', name='teacher', marker=dict(size=6, opacity=0.7)
), row=1, col=1)
max_loss = max(pp_compare_df['teacher_bfs_loss_mw'].max(), pp_compare_df['teacher_pp_loss_mw'].max())
fig_pp.add_trace(go.Scatter(x=[0, max_loss], y=[0, max_loss], mode='lines', name='1:1', line=dict(dash='dash')), row=1, col=1)
fig_pp.add_trace(go.Histogram(x=pp_compare_df['guarded_pp_loss_gap_pct'], nbinsx=40, name='guarded gap %'), row=1, col=2)
fig_pp.add_vline(x=ACCEPT_LOSS_GAP_PCT, line_dash='dash', line_color='red', row=1, col=2)
fig_pp.add_trace(go.Box(y=pp_compare_df['raw_switch_overlap'] * 100, name='raw GT'), row=1, col=3)
fig_pp.add_trace(go.Box(y=pp_compare_df['guarded_switch_overlap'] * 100, name='guarded'), row=1, col=3)
fig_pp.update_layout(title='Pandapower Validation and Switching Comparison', template='plotly_white', height=470, width=1250)
fig_pp.update_xaxes(title_text='BFS loss (MW)', row=1, col=1)
fig_pp.update_yaxes(title_text='Pandapower loss (MW)', row=1, col=1)
fig_pp.update_xaxes(title_text='Loss gap (%)', row=1, col=2)
fig_pp.update_yaxes(title_text='Switch overlap (%)', row=1, col=3)
fig_pp.show()

print('═'*78)
print('Pandapower Comparative Analysis')
print('═'*78)
print(f'  Cases compared                  : {len(pp_compare_df)}  (skipped={pp_skipped})')
print(f'  BFS vs pandapower loss diff     : mean={PP_COMPARISON_SUMMARY["pp_vs_bfs_loss_diff_mean_pct"]:.3f}%  '
      f'p95={PP_COMPARISON_SUMMARY["pp_vs_bfs_loss_diff_p95_pct"]:.3f}%')
print(f'  Raw GT vs BFS teacher loss gap  : mean={PP_COMPARISON_SUMMARY["raw_gt_bfs_loss_gap_mean_pct"]:.3f}%')
print(f'  Guarded GT vs pandapower gap    : mean={PP_COMPARISON_SUMMARY["guarded_pp_loss_gap_mean_pct"]:.3f}%  '
      f'p95={PP_COMPARISON_SUMMARY["guarded_pp_loss_gap_p95_pct"]:.3f}%')
print(f'  Raw / guarded exact switching   : {PP_COMPARISON_SUMMARY["raw_exact_switch_pct"]:.1f}% / '
      f'{PP_COMPARISON_SUMMARY["guarded_exact_switch_pct"]:.1f}%')
print(f'  Raw / guarded switch overlap    : {PP_COMPARISON_SUMMARY["raw_switch_overlap_pct"]:.1f}% / '
      f'{PP_COMPARISON_SUMMARY["guarded_switch_overlap_pct"]:.1f}%')
print(f'  Guard fallback rate             : {PP_COMPARISON_SUMMARY["fallback_rate_pct"]:.1f}%')
print(f'  Acceptance target               : mean guarded loss gap <= {ACCEPT_LOSS_GAP_PCT:.1f}%')
print(f'  Verdict                         : {"PASS" if comparison_pass else "REVIEW / TRAIN LONGER"}')
print('═'*78)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 18 — FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════
summary = pd.read_hdf(DIAG_FILE, key='summary').iloc[0]

try:
    weights = pd.read_hdf(DIAG_FILE, key='loss_weights').iloc[0]
    lam_p   = weights['lambda_physics']
    lam_v   = weights['lambda_voltage']
except:
    lam_p, lam_v = 0.10, 0.50

try:
    gt_summary = pd.read_hdf(DIAG_FILE, key='gt_training_summary').iloc[0]
except:
    gt_summary = None

try:
    pp_summary = pd.read_hdf(DIAG_FILE, key='pandapower_summary').iloc[0]
except:
    pp_summary = None

print('═'*70)
print('DNR GT + META-LEARNING PIPELINE COMPLETE')
print('═'*70)
print()
print('Network')
print(f'  IEEE 33-bus, {N_LINES} lines, {N_OPEN} tie lines')
print(f'  Z_base = {Z_BASE:.4f} Ω  (impedances converted to pu)')
print(f'  Valid spanning trees: {N_TOPOS:,}')
print()
print('Diagnostic Results')
print(f'  Scenarios run   : {int(summary.n_scenarios):,}')
print(f'  Feasible        : {int(summary.n_feasible):,}  ({summary.n_feasible/summary.n_scenarios*100:.1f}%)')
print(f'  Topology classes: {int(summary.n_classes)}')
print(f'  Loss mean±std   : {summary.loss_mean:.5f} ± {summary.loss_std:.5f} MW')
print(f'  V_min mean      : {summary.vmin_mean:.4f} pu')
print()
print('Training Config')
print(f'  Model           : Structural GraphTransformer ({GT_LAYERS} layers, {GT_HIDDEN} hidden, {GT_HEADS} heads)')
print(f'  Meta-learning   : Reptile ({GT_META_EPISODES} episodes, {GT_INNER_STEPS} inner steps)')
print(f'  Loss            : CE + {GT_PHYS_LAMBDA:.2f}*physics  (diagnostic λ≈{lam_p:.2f}, voltage λ≈{lam_v:.2f})')
print(f'  Radial guard    : GT top-{GT_CANDIDATE_K} candidates + BFS validation')
print()
if gt_summary is not None:
    print('GT Validation')
    print(f'  Line F1         : {gt_summary.val_line_f1:.3f}')
    print(f'  Exact topology  : {gt_summary.val_exact_topology*100:.1f}%')
    print(f'  Switch overlap  : {gt_summary.val_switch_overlap*100:.1f}%')
    print(f'  Loss MAPE       : {gt_summary.val_loss_mape_pct:.2f}%')
    print()
if pp_summary is not None:
    verdict = 'PASS' if bool(pp_summary.comparison_pass) else 'REVIEW / TRAIN LONGER'
    print('Pandapower Comparison')
    print(f'  Cases compared  : {int(pp_summary.n_cases)}')
    print(f'  BFS↔PP loss diff: mean={pp_summary.pp_vs_bfs_loss_diff_mean_pct:.3f}%  p95={pp_summary.pp_vs_bfs_loss_diff_p95_pct:.3f}%')
    print(f'  Guarded GT gap  : mean={pp_summary.guarded_pp_loss_gap_mean_pct:.3f}%  p95={pp_summary.guarded_pp_loss_gap_p95_pct:.3f}%')
    print(f'  Switch overlap  : {pp_summary.guarded_switch_overlap_pct:.1f}%')
    print(f'  Fallback rate   : {pp_summary.fallback_rate_pct:.1f}%')
    print(f'  Verdict         : {verdict}')
    print()
print('Stored Data')
print(f'  File    : {DIAG_FILE}')
print('  Keys    : scenarios, summary, training_matrix_check, loss_weights,')
print('            gt_training_summary, gt_training_history,')
print('            pandapower_comparison, pandapower_summary')
print('═'*70)